# Data Ingestion, Summarization and Storage

This notebook walks through the full data pipeline:

1. **Fetch & chunk** — pull Wikipedia articles and split them into ~250-word paragraphs
2. **Summarize** — call the OpenRouter API to generate three style variants per chunk (shorten / professional / informal)
3. **Inspect** — quick sanity checks on the output

The output JSONL (`data/summaries_v3.jsonl`) is the input to the training pipeline in `02_lora_laplace_training.ipynb`.

## 1. Configuration

In [ ]:
import os
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
ROOT = Path("..")
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

TITLES_FILE   = DATA_DIR / "wikipedia.lst"       # one Wikipedia title per line
CHUNKS_FILE   = DATA_DIR / "wikipedia_chunks.jsonl"
SUMMARIES_FILE = DATA_DIR / "summaries_v3.jsonl"

# ── Chunking ─────────────────────────────────────────────────────────────────
APPROX_WORDS_PER_CHUNK = 250

# ── OpenRouter ───────────────────────────────────────────────────────────────
# Set your key here or in a .env file at the project root.
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
OPENROUTER_MODEL   = "openai/gpt-4o-mini"
SUMMARY_TEMPERATURE = 0.7
SUMMARY_MAX_TOKENS  = 200
WORKERS = 4          # concurrent API threads
N_MAX   = None       # set to an int to process only the first N chunks

assert OPENROUTER_API_KEY, "Set OPENROUTER_API_KEY before running this notebook."
print("Config OK")

## 2. Wikipedia titles

Edit the list below or point `TITLES_FILE` at an existing file.

In [ ]:
TITLES = [
    "Clocks",
    "Machine learning",
    "Climate change",
    "Vaccination",
    "Quantum computing",
    "Photosynthesis",
    "History of the Internet",
    "Black hole",
    "CRISPR",
    "Ancient Rome",
]

TITLES_FILE.write_text("\n".join(TITLES) + "\n")
print(f"Wrote {len(TITLES)} titles to {TITLES_FILE}")

## 3. Fetch and chunk Wikipedia articles

In [ ]:
import sys
sys.path.insert(0, str(ROOT))

import json
import time
import ssl
import requests
from requests.adapters import HTTPAdapter
from nltk import sent_tokenize, download as nltk_download
from tqdm.notebook import tqdm

nltk_download("punkt", quiet=True)
nltk_download("punkt_tab", quiet=True)

DEFAULT_USER_AGENT = "summarizer-uncertainty-ml/0.1 (research notebook)"


def build_session():
    session = requests.Session()
    session.headers.update({"User-Agent": DEFAULT_USER_AGENT})
    return session


def fetch_wikipedia_extract(title, session, retries=3, wait=1.0):
    params = {
        "action": "query", "prop": "extracts", "explaintext": 1,
        "format": "json", "titles": title, "redirects": 1, "formatversion": 2,
    }
    for attempt in range(retries):
        try:
            r = session.get("https://en.wikipedia.org/w/api.php", params=params, timeout=30)
            r.raise_for_status()
            pages = r.json().get("query", {}).get("pages", [])
            if not pages:
                return "", None
            page = pages[0]
            pageid = page.get("pageid")
            return page.get("extract", "") or "", (
                f"https://en.wikipedia.org/?curid={pageid}" if pageid else None
            )
        except Exception as e:
            if attempt + 1 == retries:
                raise
            time.sleep(wait * (2 ** attempt))
    return "", None


def chunk_text(text, approx_words=APPROX_WORDS_PER_CHUNK):
    sents = sent_tokenize(text)
    chunks, cur, cur_words = [], [], 0
    for s in sents:
        w = len(s.split())
        if cur and cur_words + w > approx_words:
            chunks.append(" ".join(cur))
            cur, cur_words = [s], w
        else:
            cur.append(s)
            cur_words += w
    if cur:
        chunks.append(" ".join(cur))
    return chunks


session = build_session()
titles = [t.strip() for t in TITLES_FILE.read_text().splitlines() if t.strip()]
total_chunks = 0

with CHUNKS_FILE.open("w") as out:
    for title in tqdm(titles, desc="Fetching"):
        try:
            text, url = fetch_wikipedia_extract(title, session)
        except Exception as e:
            print(f"[WARN] {title}: {e}")
            continue
        if not text:
            print(f"[WARN] {title}: empty extract")
            continue
        for i, chunk in enumerate(chunk_text(text)):
            obj = {
                "id": f"wikipedia|{title.replace(' ', '_')}|chunk_{i:04d}",
                "page_title": title,
                "source_url": url,
                "paragraph_idx": i,
                "paragraph_text": chunk,
                "paragraph_word_count": len(chunk.split()),
                "fetched_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            }
            out.write(json.dumps(obj, ensure_ascii=False) + "\n")
            total_chunks += 1

print(f"\nWrote {total_chunks} chunks to {CHUNKS_FILE}")

## 4. Generate summaries via OpenRouter

Three styles per chunk: **shorten**, **professional**, **informal**.

In [ ]:
import backoff
from concurrent.futures import ThreadPoolExecutor, as_completed

OPENROUTER_ENDPOINT = "https://openrouter.ai/api/v1/chat/completions"

PROMPT_TEMPLATES = {
    "shorten": (
        "Shorten the paragraph below into 2-3 concise sentences that preserve all key facts.\n"
        "Return only the shortened text and nothing else.\n\nParagraph:\n{paragraph}"
    ),
    "professional": (
        "Write a professional 2-3 sentence summary of the paragraph below, "
        "suitable for an encyclopedia or formal report.\n"
        "Return only the summary text and nothing else.\n\nParagraph:\n{paragraph}"
    ),
    "informal": (
        "Summarize the paragraph below in 2-3 casual, conversational sentences "
        "as if explaining it to a friend.\n"
        "Return only the summary text and nothing else.\n\nParagraph:\n{paragraph}"
    ),
}


@backoff.on_exception(backoff.expo, Exception, max_tries=5)
def call_openrouter(prompt):
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": OPENROUTER_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": SUMMARY_TEMPERATURE,
        "max_tokens": SUMMARY_MAX_TOKENS,
        "n": 1,
    }
    r = requests.post(OPENROUTER_ENDPOINT, json=payload, headers=headers, timeout=60)
    if r.status_code >= 400:
        raise RuntimeError(f"OpenRouter {r.status_code}: {r.text[:500]}")
    choices = r.json().get("choices", [])
    return choices[0]["message"]["content"].strip() if choices else ""


def summarize_chunk(chunk_obj, style):
    try:
        text = call_openrouter(PROMPT_TEMPLATES[style].format(paragraph=chunk_obj["paragraph_text"]))
    except Exception as e:
        return {"error": str(e), "id": chunk_obj["id"], "summary_style": style}
    return {
        "id": f"{chunk_obj['id']}|{style}",
        "page_title": chunk_obj.get("page_title"),
        "paragraph_text": chunk_obj["paragraph_text"],
        "summary": text,
        "summary_style": style,
        "model": OPENROUTER_MODEL,
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }


chunks = []
with CHUNKS_FILE.open() as f:
    for i, line in enumerate(f):
        if N_MAX is not None and i >= N_MAX:
            break
        chunks.append(json.loads(line))

print(f"Summarizing {len(chunks)} chunks × 3 styles = {len(chunks)*3} API calls")

tasks = [(chunk, style) for chunk in chunks for style in PROMPT_TEMPLATES]
written = 0

with SUMMARIES_FILE.open("w") as out, ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futures = {pool.submit(summarize_chunk, chunk, style): (chunk["id"], style)
               for chunk, style in tasks}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Summarizing"):
        result = future.result()
        if "error" in result:
            print(f"[ERROR] {result}")
        else:
            out.write(json.dumps(result, ensure_ascii=False) + "\n")
            written += 1

print(f"\nWrote {written} summaries to {SUMMARIES_FILE}")

## 5. Inspect results

In [ ]:
import pandas as pd

records = [json.loads(l) for l in SUMMARIES_FILE.open() if l.strip()]
df = pd.DataFrame(records)

print(f"Total records : {len(df)}")
print(f"Styles        : {df['summary_style'].value_counts().to_dict()}")
print(f"Pages         : {df['page_title'].nunique()}")
print()
print("Source length (words):")
df["src_words"] = df["paragraph_text"].str.split().str.len()
print(df["src_words"].describe().round(1))
print()
print("Summary length (words):")
df["tgt_words"] = df["summary"].str.split().str.len()
print(df["tgt_words"].describe().round(1))

In [ ]:
# Show one example per style for the first chunk
first_base = df["id"].str.rsplit("|", n=1).str[0].iloc[0]
sample = df[df["id"].str.startswith(first_base)]

for _, row in sample.iterrows():
    print(f"=== {row['summary_style'].upper()} ===")
    print(f"SOURCE : {row['paragraph_text'][:200]}...")
    print(f"SUMMARY: {row['summary']}")
    print()